In [1]:
import pandas as pd

In [2]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 387.5 kB/s  0:00:56m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 1.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 745.5 kB/s  0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [wandb]m 9/10 [wandb]ic]-types]i]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
API_KEY = "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"

In [9]:
!wandb login "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"

wandb: ERROR Find detailed error logs at: /tmp/debug-cli.qs-ashish.log
Error: Failed to authenticate with https://api.wandb.ai: Failed to initialize API resources: the service process is busy and did not respond in time.


In [6]:
x = input("enter 5")

In [1]:
import wandb

In [3]:
wandb.login(API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/qs-ashish/.netrc
wandb: Currently logged in as: 220ashish0019 (dbit_personal_projects) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import tensorflow as tf
import matplotlib.pyplot as plt

I0000 00:00:1788187053.325636    7873 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788187053.337428    7873 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788187053.379657    7873 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788187054.922733    7873 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [6]:

sweep_config = {
    'method': 'grid',

    'metric': {
        'name': 'val_accuracy',
        'goal': 'maximize',
    },

    'parameters': {
        'batch_size': {
            'values': [8, 16],
        },

        'learning_rate': {
            'values': [0.001, 0.01],
        },

        'hidden_nodes': {
            'values': [128, 64],
        },

        'img_size': {
            'values': [16, 224],
        },

        'epochs': {
            'values': [5, 10],
        },
    },
}


In [7]:

# Create the sweep
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project='dbit_personal_projects'
)

print("Sweep ID:", sweep_id)

Create sweep with ID: eidxiusx
Sweep URL: https://wandb.ai/dbit_personal_projects/dbit_personal_projects/sweeps/eidxiusx
Sweep ID: eidxiusx


In [8]:
def get_dataset_partitions(ds, train_size=0.8, val_size=0.1, test_size=0.1, shuffle=True, shuffle_size=100):
    assert(train_size + val_size + test_size == 1)


    ds_size = len(ds)

    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)

    train_size = int(train_size * ds_size)
    val_size = int(val_size * ds_size)
    test_size = int(test_size * ds_size)

    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(test_size).skip(val_size).take(test_size)

    return train_ds, val_ds, test_ds

In [9]:
from tensorflow import  keras

In [10]:
def train():
    with wandb.init() as run:

        config = wandb.config
        dataset_path = "./datasets/flower_photos/"

        IMG_SIZE = config.img_size
        IMG_CHANNELS = 3
        BATCH_SIZE = 32
        CHANNELS = 3
        EPOCHS = config.epochs
        CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

        dataset = tf.keras.preprocessing.image_dataset_from_directory(
            dataset_path,
            shuffle = True,
            image_size = (IMG_SIZE,IMG_SIZE), # height and width
            batch_size = BATCH_SIZE,
        )

        train_ds, val_ds, test_ds  = get_dataset_partitions(dataset)

        train_ds.cache().shuffle(100).prefetch(buffer_size = tf.data.AUTOTUNE)
        val_ds.cache().shuffle(100).prefetch(buffer_size = tf.data.AUTOTUNE)
        test_ds.cache().shuffle(100).prefetch(buffer_size = tf.data.AUTOTUNE)

        model = keras.Sequential([
            keras.layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS)),
            keras.layers.Dense(config.hidden_nodes, activation='relu'),
            keras.layers.Dense(len(CLASS_NAMES), activation="softmax")
        ])

        model.compile(
            optimizer="adam",
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
            metrics=["accuracy"]
        )

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS,
            callbacks=[WandbMetricsLogger(log_freq=5)]
        )

In [11]:
wandb.agent(sweep_id, function=train)

wandb: Agent Starting Run: w4bogdnl with config:
wandb: 	batch_size: 8
wandb: 	epochs: 5
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/qs-ashish/.netrc.
wandb: setting up run w4bogdnl
wandb: Tracking run with wandb version 0.29.0
wandb: Run data is saved locally in /media/qs-ashish/DATA/ashish/core-python/wandb/run-20260831_205056-w4bogdnl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lyric-sweep-1
wandb: ⭐️ View project at https://wandb.ai/dbit_personal_projects/dbit_personal_projects
wandb: 🧹 View sweep at https://wandb.ai/dbit_personal_projects/dbit_personal_projects/sweeps/eidxiusx
wandb: 🚀 View run at https://wandb.ai/dbit_personal_projects/dbit_personal_projects/runs/w4bogdnl


Found 3670 files belonging to 5 classes.


E0000 00:00:1788189662.046994   11078 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1788189662.047241   11145 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1788189662.079811   11078 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/5


/media/qs-ashish/DATA/ashish/core-python/.venv/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1788189673.458752   11176 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 14 of 100
I0000 00:00:1788189693.871910   11176 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 44 of 100
I0000 00:00:1788189714.052633   11176 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 73 of 100
I0000 00:00:1788189724.089777   11176 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 87 of 100
I0000 00:00:1788189733.190511   11176 shuffle_dataset_op.cc:483] Shuffle buffer fille

...

Found 3670 files belonging to 5 classes.


/media/qs-ashish/DATA/ashish/core-python/.venv/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
19/92 ━━━━━━━━━━━━━━━━━━━━ 11s 163ms/step - accuracy: 0.2175 - loss: 17107.8961

wandb: Ctrl + C detected. Stopping sweep.


23/92 ━━━━━━━━━━━━━━━━━━━━ 11s 161ms/step - accuracy: 0.2210 - loss: 15893.6514